# SpaNCy-Shift DDPM — averaged SDEdit + GNN noise controls

Copy of `spancy_shift_ddpm_eval.ipynb` (2026-10-02). That run passed every test except **4b**:
per-cell output was mostly random (seed SD / |delta| = 0.75), so the same cell gets a different
correction on every run. It also showed that **Stage 1 + matched Gaussian noise alone scores
kBET 0.696** — kBET rewards blur. This notebook asks two questions:

**A. Does DDPM keep its kBET once the randomness is averaged away?** Each output is the mean of
K independent SDEdit draws from the same trained model (lean run: K = 1, 4; 2 independent averages
per K, disjoint draws → 8 draws). Averaging cuts per-cell noise by about √K (K=4 halves it).
One draw takes ~7 min on a T4 (compute-bound; measured 2026-10-02), so the full K = 1/4/8 × 3-rep
run (24 draws, ~3 h) is left as constants at the top of §4a.
- K=4 kBET stays near 0.73 and 4b passes → the gain is real alignment; DDPM is a headline candidate.
- K=4 kBET falls toward the GNN / noise-control level → much of DDPM's extra kBET was blur.

**B. Does the published GNN pass the same noise control DDPM passed?** The GNN (published config,
`w_adv=0.3`, α=0.6, 10 epochs) is trained at seeds 0/1/2. Each seed gets its own Stage 1 + Gaussian
noise control, SD matched per batch and marker to that seed's correction; seed 0 also gets the
shuffled control and a 3-run inference stability check.

Biology (positive population, 1D shape, silhouette) is computed for Stage 1, DDPM K=1, DDPM K=`K_MAIN`
and GNN seed 0. kBET is computed for every layer (12 layers × 5 groups in the lean run).

**To run:** Runtime → Run all (GPU runtime). Upload **`spancy_shift.py`** and **`spancy_shift_ddpm.py`**
when §0 asks. If RAM runs short, switch to a High-RAM runtime (~12 layers of 141 MB each).
Paste back §4a (averaging stability), §4b (GNN stability), §6 (positive population summary lines),
§6b (unimodal summary table), §8 (kBET summary + per-group) + silhouette Mean row, and **§10 (verdict)**.

## 0. Colab Setup

torch-geometric is needed for the GNN arm (§4b).

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version) — needed for the GNN arm
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload spancy_shift.py AND spancy_shift_ddpm.py (only missing files are requested).
from google.colab import files
import os

NEEDED = ['spancy_shift.py', 'spancy_shift_ddpm.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()

for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:24s} {os.path.getsize(f):>10,} bytes')

In [ ]:
import importlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift
import spancy_shift_ddpm
from spancy_shift import (
    load_adata, log1p_scale,
    detect_bimodal_markers, find_best_sample_per_marker,
    shift_normalize_per_marker,
    positive_population_table, summarize_positive_population,
    per_marker_batch_r2,
    train as train_gnn, normalize_adata as normalize_adata_gnn,
)
from spancy_shift_ddpm import (
    DenoisingMLP, DDPMScheduler, BatchBalancedSampler,
    train_ddpm, normalize_adata_ddpm,
)

# After re-uploading files:
# importlib.reload(spancy_shift); importlib.reload(spancy_shift_ddpm)
# from spancy_shift import *; from spancy_shift_ddpm import *

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load & Inspect Data

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

## 2. Bimodal Marker Detection

Bimodal markers are **excluded from DDPM normalization** — they pass through Stage 1 output
unchanged. Shown here so you know which markers the DDPM is and isn't correcting.

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: DDPM scores the full joint distribution — no special handling required.")
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

## 2b. Reference Sample Selection

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)

ref_df = pd.DataFrame(
    [(m, s) for m, s in ref_sample_per_marker.items()],
    columns=['marker', 'reference_sample'],
)
print('Per-marker reference sample:\n')
print(ref_df.to_string(index=False))

ref_counts = ref_df['reference_sample'].value_counts()
print(f'\nDistinct reference samples: {len(ref_counts)}')

fig, ax = plt.subplots(figsize=(14, 3))
sample_order = sorted(adata.obs['sample_id'].unique())
ref_matrix = [sum(v == s for v in ref_sample_per_marker.values()) for s in sample_order]
ax.bar(range(len(sample_order)), ref_matrix, color='steelblue', alpha=0.85)
ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(sample_order, rotation=45, ha='right')
ax.set_ylabel('# markers using as reference')
ax.set_title('Reference sample selection (medoid)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 3. Training (DDPM)

**Inside `train_ddpm()`:**
1. Runs `shift_normalize_per_marker()` → `X_base` (Stage 1)
2. Fits `RobustScaler` on `log1p(X_base)` → `X_scaled`
3. Identifies reference batch (majority vote over per-marker reference samples)
4. Standard DDPM training loop:
   - Sample random timestep `t ~ Uniform(1, T)`
   - Add noise: `x_t = sqrt(ᾱ_t)*x_0 + sqrt(1-ᾱ_t)*eps`
   - Classifier-free guidance: randomly replace batch label with null token (cfg_dropout=10%)
   - Predict `eps`; Loss = MSE(eps_pred, eps)

**Training should be very stable** — simple MSE regression, no adversarial dynamics.
Loss should decrease monotonically from ~1.0 toward ~0.1–0.3.

In [ ]:
import gc

T_DIFFUSION = 200   # total DDPM timesteps (fixed)
T_INFER = 30        # SDEdit noise level (as in the original notebook)
CFG_SCALE = 1.5     # CFG guidance strength (as in the original notebook)
N_EPOCHS = 10       # same budget as the GNN / CFM runs

TRAIN_SEED = 0      # pins model init + training noise (the sampler is seeded internally)
torch.manual_seed(TRAIN_SEED)
torch.cuda.manual_seed_all(TRAIN_SEED)

model, scheduler, trained_scaler, trained_ref, history = train_ddpm(
    adata,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    device_str=DEVICE,
    warmup_epochs=2,
    n_per_batch=512,
    T=T_DIFFUSION,
    cfg_dropout=0.1,
    hidden=512,
    n_layers=6,
    time_dim=256,
    batch_emb_dim=32,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
    ref_sample_per_marker=ref_sample_per_marker,
)

print(f'\nTraining complete.')
print(f'  Reference batch: {model._ref_batch_name} (code {model._ref_batch_code})')
print(f'  Final loss: {history["loss"][-1]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['loss'], color='k', linewidth=1.5)
axes[0].set_title('DDPM Denoising Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss (noise prediction)')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(bottom=0)

axes[1].plot(history['lr'], color='purple', linewidth=1.5)
axes[1].set_title('Learning Rate')
axes[1].set_xlabel('Epoch')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Stage 2 Training — DenoisingMLP (DDPM)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nExpected: loss starts near 1.0 (random noise prediction), decreases to 0.1–0.3.')
print(f'If loss stays flat at ~1.0, model is not learning — try more epochs or smaller lr.')

## 4a. DDPM — average K SDEdit draws

The trained model is run through SDEdit `N_REPS × max(K)` times (lean run: 8; draw *d* uses noise
seed *d*). Rep *r* of K averages draws `r·K … (r+1)·K − 1`, so the reps of each K are independent.
K=1 reps are draws 0/1/2. Draw 0 is bit-identical to the module at seed 0; draws 1+ run in larger GPU
chunks (`DRAW_BATCH`, for speed), so they are valid draws but not bit-identical to the eval
notebook's seeds 1/2. The K=1 stability ratio (0.75 there) should still reproduce statistically.

Averaging happens in the model's scaled space before the inverse transform. That space is an
affine map of log1p, so it is the same as averaging in log1p.

`sdedit_scaled()` below is the inference loop of `normalize_adata_ddpm` copied verbatim (no Stage 1
rerun per draw). Draw 0 is checked against the module's own output at seed 0.

**Stability metric (4b)**: mean per-cell SD across the reps / mean |delta| of rep 0, unimodal
markers, log1p space. 0 = deterministic; ≥ 0.5 = much of each cell's correction is random.

In [ ]:
import time

K_LIST = [1, 4]      # SDEdit draws averaged per output. Lean run: one draw is ~7 min on a T4
N_REPS = 2           # independent averages per K (disjoint draws); needs >= 2 for 4b
K_MAIN = 4           # the averaged DDPM judged in §10 (full run: K_LIST=[1,4,8], N_REPS=3, K_MAIN=8)
DRAW_BATCH = 65536   # cells per GPU chunk for draws 1+. The module's 4096 leaves the GPU idle
                     # (~8 min/draw). Same math; only the RNG order differs, so draw 0 stays at
                     # 4096 to keep the exact check against the module.

# ---- module reference run at seed 0 (also produces Stage 1 = normalized_base) --
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
adata_norm = normalize_adata_ddpm(
    adata, model, scheduler, trained_scaler, trained_ref,
    t_infer=T_INFER, cfg_scale=CFG_SCALE,
    inference_batch_size=4096, device_str=DEVICE,
    layer_name='normalized', keep_base_layer=True,
)
X_base = np.asarray(adata_norm.layers['normalized_base'])
base_log = np.log1p(np.clip(X_base, 0, None)).astype(np.float32)
X_scaled = trained_scaler.transform(base_log).astype(np.float32)   # same as the module

# ---- one SDEdit draw (verbatim copy of normalize_adata_ddpm's loop) ----------
_dev = torch.device(DEVICE)
_bimodal_dims = torch.tensor(getattr(model, '_is_bimodal', marker_is_bimodal),
                             dtype=torch.bool, device=_dev)
_sqrt_ab = scheduler.sqrt_alphas_bar[T_INFER - 1]
_sqrt_1mab = scheduler.sqrt_one_minus_alphas_bar[T_INFER - 1]


@torch.no_grad()
def sdedit_scaled(seed, batch_size=4096):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    out = np.empty_like(X_scaled)
    n_b, ref_code = model._n_batches, model._ref_batch_code
    for start in range(0, len(X_scaled), batch_size):
        end = min(start + batch_size, len(X_scaled))
        x_0 = torch.tensor(X_scaled[start:end], device=_dev)
        B = len(x_0)
        x_t = _sqrt_ab * x_0 + _sqrt_1mab * torch.randn_like(x_0)
        ref_ids = torch.full((B,), ref_code, dtype=torch.long, device=_dev)
        null_ids = torch.full((B,), n_b, dtype=torch.long, device=_dev)
        for t_step in range(T_INFER, 0, -1):
            t_tensor = torch.full((B,), t_step, dtype=torch.long, device=_dev)
            if CFG_SCALE == 1.0:
                eps_pred = model(x_t, t_tensor, ref_ids)
            else:
                eps_cond = model(x_t, t_tensor, ref_ids)
                eps_uncond = model(x_t, t_tensor, null_ids)
                eps_pred = eps_uncond + CFG_SCALE * (eps_cond - eps_uncond)
            x_t = scheduler.p_step(x_t, t_step, eps_pred, add_noise=(t_step > 1))
            x_t[:, _bimodal_dims] = x_0[:, _bimodal_dims]
        out[start:end] = x_t.cpu().numpy()
    return out


def to_counts(x_scaled):
    return np.clip(np.expm1(trained_scaler.inverse_transform(x_scaled)), 0, None).astype(np.float32)


def to_log(x_scaled):
    return np.log1p(to_counts(x_scaled))


# ---- 24 draws, accumulated into disjoint K-averages --------------------------
n_draws = N_REPS * max(K_LIST)
acc = {K: [np.zeros_like(X_scaled) for _ in range(N_REPS)] for K in K_LIST}
t0 = time.time()
for d in range(n_draws):
    xs = sdedit_scaled(d, batch_size=4096 if d == 0 else DRAW_BATCH)
    if d == 0:
        diff = np.abs(np.log1p(to_counts(xs)) -
                      np.log1p(np.clip(np.asarray(adata_norm.layers['normalized']), 0, None)))
        print(f'Check vs module seed 0: max |log1p diff| = {diff.max():.2e}  '
              f'(~0 = helper reproduces the module exactly)')
        if diff.max() > 1e-3:
            print('  WARNING: helper differs from the module (RNG order?). Draws are still valid SDEdit')
            print('  samples, but K=1 will not reproduce the eval notebook exactly.')
        del diff
    for K in K_LIST:
        r = d // K
        if r < N_REPS:
            acc[K][r] += xs
    del xs
    print(f'  draw {d + 1}/{n_draws}  ({time.time() - t0:.0f}s)')
gc.collect()

# ---- stability per K + store layers ------------------------------------------
uni_markers = [m for m in marker_names if m not in bimodal_names]
uni_idx = [marker_names.index(m) for m in uni_markers]
STAB = {}
for K in K_LIST:
    logs = [to_log(a / K) for a in acc[K]]
    d0 = logs[0] - base_log
    d1 = logs[1] - base_log
    sd = np.std(np.stack(logs), axis=0)
    STAB[K] = {
        'mean_abs_delta': float(np.abs(d0[:, uni_idx]).mean()),
        'mean_seed_sd': float(sd[:, uni_idx].mean()),
        'corr_rep0_rep1': float(np.corrcoef(d0[:, uni_idx].ravel(), d1[:, uni_idx].ravel())[0, 1]),
    }
    STAB[K]['ratio'] = STAB[K]['mean_seed_sd'] / STAB[K]['mean_abs_delta']
    if K != 1:
        reps = range(N_REPS) if K == K_MAIN else [0]
        for r in reps:
            adata_norm.layers[f'ddpm_k{K}_r{r}'] = np.clip(np.expm1(logs[r]), 0, None).astype(np.float32)
    if K == K_MAIN:
        delta_main = d0.copy()
    del logs, d0, d1, sd, acc[K]
    gc.collect()

stab_df = pd.DataFrame(STAB).T.rename_axis('K')
print('\n' + '=' * 78)
print('4a. STABILITY vs K (unimodal markers, log1p space)')
print('=' * 78)
print(stab_df[['mean_abs_delta', 'mean_seed_sd', 'ratio', 'corr_rep0_rep1']].round(4).to_string())
print('\nratio = mean seed SD / mean |delta|  (4b needs < 0.5; eval notebook K=1: 0.75)')
print('corr_rep0_rep1 = how similar two independent averages correct the same cell (1 = identical)')
print('mean_abs_delta shrinking with K = the averaged-away part of the correction was noise')

# ---- noise controls matched to the K_MAIN rep-0 correction -------------------
rng = np.random.default_rng(0)
batches_arr = adata_norm.obs['batch_id'].astype(str).values


def add_controls(delta, key_gauss, key_perm, rng):
    """Stage 1 + Gaussian noise (SD matched per batch and marker) / + shuffled delta."""
    g = base_log.copy()
    p = base_log.copy() if key_perm else None
    for b in np.unique(batches_arr):
        ii = np.where(batches_arr == b)[0]
        db = delta[ii]
        g[ii] += rng.standard_normal(db.shape).astype(np.float32) * db.std(axis=0)
        if p is not None:
            p[ii] += db[rng.permutation(len(ii))]
    adata_norm.layers[key_gauss] = np.clip(np.expm1(g), 0, None).astype(np.float32)
    if p is not None:
        adata_norm.layers[key_perm] = np.clip(np.expm1(p), 0, None).astype(np.float32)


add_controls(delta_main, 'dctrl_gauss', 'dctrl_perm', rng)
del delta_main, X_scaled
gc.collect()
print(f'\nLayers: {list(adata_norm.layers.keys())}')

## 4b. GNN (published config) at 3 seeds + its own noise controls

Same training call as `spancy_shift_mmdresnet_s1_wadv03.ipynb` (old graph, `w_adv=0.3`, α=0.6,
10 epochs). Seed 0 is also run through inference 3 times to measure the GNN's per-cell
stability with the same ratio as 4a. Each seed's Gaussian control is matched to that seed's own
correction, so the comparison in §10 is paired per seed.

In [ ]:
HYBRID_ALPHA = 0.6
K_NEIGHBORS = 15
GNN_SEEDS = [0, 1, 2]

gnn_inf_logs = []
for seed in GNN_SEEDS:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    m_s, sc_s, ref_s, h_s = train_gnn(
        adata, n_epochs=10, lr=1e-3, device_str=DEVICE, warmup_epochs=2,
        n_per_batch=512, k_neighbors=K_NEIGHBORS, hybrid_alpha=HYBRID_ALPHA,
        temperature=0.07, w_recon=0.1, w_contrast=0.5, w_adv=0.3, w_mmd=1.0,
        mmd_samples=256, grl_max=1.0, bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
        ref_sample_per_marker=ref_sample_per_marker,
    )
    n_inf = 3 if seed == GNN_SEEDS[0] else 1
    for j in range(n_inf):
        torch.manual_seed(100 + j)
        torch.cuda.manual_seed_all(100 + j)
        out = normalize_adata_gnn(
            adata, m_s, sc_s, ref_s, hybrid_alpha=HYBRID_ALPHA, k_neighbors=K_NEIGHBORS,
            inference_batch_size=2048, device_str=DEVICE,
            layer_name='normalized', keep_base_layer=True,
        )
        if seed == GNN_SEEDS[0] and j == 0:
            s1_diff = np.abs(np.asarray(out.layers['normalized_base']) - X_base).max()
            print(f'Stage 1 check (GNN path vs DDPM path): max |diff| = {s1_diff:.2e}  (expect 0)')
        g = np.asarray(out.layers['normalized'], dtype=np.float32)
        if j == 0:
            adata_norm.layers[f'gnn_s{seed}'] = g
        if seed == GNN_SEEDS[0]:
            gnn_inf_logs.append(np.log1p(np.clip(g, 0, None)))
        del out, g
        gc.collect()

    delta = np.log1p(np.clip(adata_norm.layers[f'gnn_s{seed}'], 0, None)) - base_log
    add_controls(delta, f'gctrl_gauss_s{seed}',
                 f'gctrl_perm_s{seed}' if seed == GNN_SEEDS[0] else None,
                 np.random.default_rng(seed))
    print(f'GNN seed {seed}: mean |delta| (unimodal) = {np.abs(delta[:, uni_idx]).mean():.4f}')
    del m_s, sc_s, ref_s, h_s, delta
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

d0 = gnn_inf_logs[0] - base_log
sd = np.std(np.stack(gnn_inf_logs), axis=0)
GNN_STAB = float(sd[:, uni_idx].mean() / np.abs(d0[:, uni_idx]).mean())
print('\n' + '=' * 78)
print(f'4b. GNN per-cell stability (seed 0, 3 inference runs): seed SD / |delta| = {GNN_STAB:.4f}')
print(f'    DDPM for comparison: K=1 {STAB[1]["ratio"]:.2f}, K={K_MAIN} {STAB[K_MAIN]["ratio"]:.2f}')
print('=' * 78)
del gnn_inf_logs, d0, sd
gc.collect()

In [ ]:
# Canonical layer lists. ABL_LAYERS -> kBET (all). BIO_LAYERS -> pos-pop, shape, silhouette.
ABL_LAYERS = [('normalized_base', 'Stage 1'), ('normalized', 'DDPM K1')]
ABL_LAYERS += [(f'ddpm_k{K}_r0', f'DDPM K{K} r0') for K in K_LIST if K != 1]
ABL_LAYERS += [(f'ddpm_k{K_MAIN}_r{r}', f'DDPM K{K_MAIN} r{r}') for r in range(1, N_REPS)]
ABL_LAYERS += [('dctrl_gauss', f'DDPM-K{K_MAIN} S1+noise'),
               ('dctrl_perm', f'DDPM-K{K_MAIN} S1+shuffled')]
ABL_LAYERS += [(f'gnn_s{s}', f'GNN s{s}') for s in GNN_SEEDS]
ABL_LAYERS += [(f'gctrl_gauss_s{s}', f'GNN-s{s} S1+noise') for s in GNN_SEEDS]
ABL_LAYERS += [(f'gctrl_perm_s{GNN_SEEDS[0]}', f'GNN-s{GNN_SEEDS[0]} S1+shuffled')]
ABL_LABEL = dict(ABL_LAYERS)

BIO_LAYERS = [('normalized_base', 'Stage 1'), ('normalized', 'DDPM K1'),
              (f'ddpm_k{K_MAIN}_r0', f'DDPM K{K_MAIN} r0'),
              (f'gnn_s{GNN_SEEDS[0]}', f'GNN s{GNN_SEEDS[0]}')]

missing = [k for k, _ in ABL_LAYERS if k not in adata_norm.layers]
assert not missing, f'missing layers: {missing}'
print(f'{len(ABL_LAYERS)} kBET layers, {len(BIO_LAYERS)} biology layers')

## 6. Positive Population Preservation Check

% positive cells per marker per sample using **per-sample GMM** (2-component Gaussian mixture
fitted on each sample's raw data in log10 space; threshold applied to both raw and normalized
for that sample). Delta = normalized − raw. Target: |delta| < 5% per marker.

**Headline metric**: the full per-marker mean Δ (± SD) table over all 20 markers, sorted by
mean Δ, with a `pass_5pct` column and a count of markers within ±5%. This is the primary
biology-preservation result. A high SD (CD45, CDX2, DAPI_R1, ...) flags a unimodal marker whose
GMM threshold sits near the histogram peak — its large |Δ| is threshold instability, not real
biology distortion. Read mean Δ alongside its SD; do **not** drop markers.

The density-at-threshold reliability filter (`summarize_positive_population()`) is kept **for
reference only** at the end of the cell — it drops most unimodal markers (often leaving ~3),
too few to compare methods, so it is not the primary metric.

In [ ]:
POP_LAYERS = BIO_LAYERS   # biology arms only (controls skipped)

# Per-sample GMM positive-population table for every arm.
pop_tables_abl = {}
for layer_key, label in POP_LAYERS:
    pop_tables_abl[label] = positive_population_table(
        adata_norm, norm_layer=layer_key, sample_col='sample_id'
    )

# Keep the original names so the heatmap / scatter / per-sample breakdown cells below
# keep working unchanged (they show the 'full' arm).
pop_table_base = pop_tables_abl['Stage 1']
pop_table = pop_tables_abl[POP_LAYERS[1][1]]


# ── HEADLINE: full per-marker table, ALL 20 markers (primary biology metric) ──
# mean Δ = mean over samples of (norm − raw) % positive. SD = inter-sample spread.
# High SD (CD45, CDX2, DAPI_R1, ...) flags a unimodal marker whose GMM threshold sits
# near the histogram peak → the large |Δ| reflects threshold instability, NOT real
# biology distortion. Read mean Δ alongside its SD; do not filter markers out.
def pop_summary(pt):
    s = pt.groupby('marker').agg(
        mean_pct_raw=('pct_pos_raw', 'mean'),
        mean_pct_norm=('pct_pos_norm', 'mean'),
        mean_delta=('delta', 'mean'),
        sd_delta=('delta', 'std'),
        mean_abs_delta=('delta', lambda x: x.abs().mean()),
    ).round(2)
    s['pass_5pct'] = s['mean_delta'].abs() < 5
    return s.sort_values('mean_delta')


summaries = {label: pop_summary(pt) for label, pt in pop_tables_abl.items()}

# ---- cross-arm comparison (the ablation view) ------------------------------
delta_cmp = pd.DataFrame({label: s['mean_delta'] for label, s in summaries.items()})
sd_cmp = pd.DataFrame({label: s['sd_delta'] for label, s in summaries.items()})
delta_cmp = delta_cmp.reindex(summaries['Stage 1'].index)
sd_cmp = sd_cmp.reindex(delta_cmp.index)

print('=' * 95)
print('POSITIVE POPULATION — mean delta % per marker, per layer (target |delta| < 5%)')
print('=' * 95)
print(delta_cmp.round(2).to_string())
print('\nInter-sample SD of delta (high SD = unimodal marker, threshold near the peak):')
print(sd_cmp.round(2).to_string())

print('\nMarkers within +/-5% mean delta:')
for label, s in summaries.items():
    print(f'  {label:>10s}: {int(s["pass_5pct"].sum())}/{len(s)}   '
          f'overall mean |delta| = {pop_tables_abl[label]["delta"].abs().mean():.2f}%')
print('\nIf the arms are within ~1pp of each other here, the dropped loss does not affect')
print('1D positive-population biology — decide on kBET / silhouette / shape instead.')

# ---- full per-arm tables ---------------------------------------------------
for label, s in summaries.items():
    print('\n' + '=' * 95)
    print(f'Positive Population Preservation — {label} vs raw  (ALL 20 markers):')
    print('=' * 95)
    print(s.to_string())
    print(f'\n  Markers within +/-5% mean delta: {int(s["pass_5pct"].sum())}/{len(s)}')

print('\nTarget: |mean delta| < 5% per marker. High-SD failing markers are unimodal (threshold')
print('near the histogram peak) — interpret their |delta| as threshold instability, not distortion.')

# ── Secondary (reference only): reliability-filtered view ─────────────────────
# The density-at-threshold filter drops most unimodal markers — often leaving only
# ~3 markers, too few to compare methods. Kept for reference; NOT the headline metric.
print('\n' + '─' * 95)
print('Secondary (reference only) — reliability-filtered; drops most markers, not primary:')
print('─' * 95)
for label, pt in pop_tables_abl.items():
    _ = summarize_positive_population(pt, min_reliable_frac=0.5, label=label)

In [ ]:
# ============================================================================
# 6b. SHAPE-PRESERVATION DIAGNOSTIC  (location-invariant)  — per layer
# ----------------------------------------------------------------------------
# Positive-pop Δ (fraction over a threshold) and silhouette (3-blob 20D
# separation) are both SHAPE-BLIND: you can flatten a sharp peak into a broad
# smear and still pass both. This cell measures whether each marker's 1D
# marginal SHAPE survives, INDEPENDENT of any location shift Stage 1 / GNN apply.
#
# Per (marker, sample), in log1p space, normalized vs raw (1.0 = shape preserved):
#   peak_ratio = peak density(norm) / peak density(raw)   <1 => mode flattened
#   var_ratio  = var(norm) / var(raw)                     >1 => broadened
#   iqr_ratio  = IQR(norm) / IQR(raw)                     >1 => broadened (robust)
#
# peak_ratio uses SHARED bin edges (union range) so it is binning-fair.
# Stage 1 is a pure translation -> should sit ~1.0 everywhere (control).
# NOTE: bimodal markers (ECAD etc.) are masked from the GNN's MMD loss, so their
# delta stays ~0 and they trivially score ~1.0 for every arm — they prove nothing
# about the ablation. Judge the arms on the unimodal markers the GNN corrects.
#
# ABLATION READING: if an arm drops a loss and peak/var stay ~1.0 while kBET holds,
# that loss was not buying shape safety.
# ============================================================================
from scipy.stats import iqr as _iqr

SHAPE_LAYERS = globals().get('BIO_LAYERS',
                             [('normalized_base', 'Stage 1'), ('normalized', 'full')])
SHAPE_LABELS = [label for _, label in SHAPE_LAYERS]
S1_LABEL = SHAPE_LABELS[0]                                  # control (pure shift)
MAIN_LABEL = SHAPE_LABELS[1] if len(SHAPE_LABELS) > 1 else S1_LABEL
N_SHAPE_BINS = 80
MIN_CELLS    = 50

# --- precompute ONCE ---
sample_ids_shape = adata_norm.obs['sample_id'].values
sample_unique    = np.unique(sample_ids_shape)
sample_idx       = {s: np.where(sample_ids_shape == s)[0] for s in sample_unique}
sample_idx       = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log        = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


rows = []
for layer_key, layer_label in SHAPE_LAYERS:
    X_layer_log = np.log1p(np.clip(np.asarray(adata_norm.layers[layer_key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        raw_col  = X_raw_log[:, mi]
        norm_col = X_layer_log[:, mi]
        peaks, vrs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(raw_col[idx], norm_col[idx])
            if np.isfinite(p):  peaks.append(p)
            if np.isfinite(v):  vrs.append(v)
            if np.isfinite(ir): irs.append(ir)
        rows.append({
            'method': layer_label, 'marker': mname,
            'peak_ratio': np.mean(peaks) if peaks else np.nan,
            'var_ratio':  np.mean(vrs)  if vrs  else np.nan,
            'iqr_ratio':  np.mean(irs)  if irs  else np.nan,
        })
    del X_layer_log
shape_df = pd.DataFrame(rows)

# ---- side-by-side table: every arm per marker ------------------------------
piv = shape_df.pivot(index='marker', columns='method',
                     values=['peak_ratio', 'var_ratio', 'iqr_ratio']).round(3)
piv = piv.reindex(columns=pd.MultiIndex.from_product(
    [['peak_ratio', 'var_ratio', 'iqr_ratio'], SHAPE_LABELS]))
piv = piv.sort_values(('peak_ratio', MAIN_LABEL))

print('=' * 92)
print('SHAPE-PRESERVATION DIAGNOSTIC  (1.0 = shape preserved; per-sample mean vs raw, log1p)')
print('  peak_ratio <1 = mode flattened   |   var_ratio / iqr_ratio >1 = broadened')
print('=' * 92)
print(piv.to_string())

by_method = {label: shape_df[shape_df.method == label].set_index('marker')
             for label in SHAPE_LABELS}
s1 = by_method[S1_LABEL]
PEAK_LO, VAR_HI, VAR_LO = 0.80, 1.25, 0.80

# ---- per-arm distortion flags ----------------------------------------------
print('\n' + '=' * 92)
print(f'SHAPE-DISTORTED markers per arm (peak<{PEAK_LO} or var outside [{VAR_LO},{VAR_HI}])')
print('=' * 92)
for label in SHAPE_LABELS[1:]:
    arm = by_method[label]
    distorted = arm[(arm.peak_ratio < PEAK_LO) |
                    (arm.var_ratio > VAR_HI) | (arm.var_ratio < VAR_LO)]
    print(f'\n{label}:  {len(distorted)} distorted')
    if len(distorted) == 0:
        print('  none — preserves 1D shape on all markers')
    else:
        for m, r in distorted.sort_values('peak_ratio').iterrows():
            tag = ' [bimodal/passed-through]' if m in bimodal_names else ''
            print(f'  {m:>10s}  peak={r.peak_ratio:.2f} (S1 {s1.loc[m,"peak_ratio"]:.2f})  '
                  f'var={r.var_ratio:.2f} (S1 {s1.loc[m,"var_ratio"]:.2f})  '
                  f'iqr={r.iqr_ratio:.2f}{tag}')

# ---- unimodal-only summary across arms -------------------------------------
uni = [m for m in s1.index if m not in bimodal_names]
uni_rows = []
for label in SHAPE_LABELS:
    arm = by_method[label]
    flagged = arm.loc[uni]
    uni_rows.append({
        'arm': label,
        'mean_peak_ratio': round(float(flagged['peak_ratio'].mean()), 3),
        'mean_var_ratio':  round(float(flagged['var_ratio'].mean()), 3),
        'mean_iqr_ratio':  round(float(flagged['iqr_ratio'].mean()), 3),
        'n_distorted': int(((flagged.peak_ratio < PEAK_LO) |
                            (flagged.var_ratio > VAR_HI) |
                            (flagged.var_ratio < VAR_LO)).sum()),
    })
print('\n' + '=' * 92)
print('Unimodal markers only (the ones DDPM actually corrects) — 1.0 = shape preserved')
print('=' * 92)
print(pd.DataFrame(uni_rows).to_string(index=False))
print(f'\n{S1_LABEL} is the control: a pure translation, so it should sit at ~1.0.')
print('An arm that matches it has not bought its kBET by reshaping marginals.')

# ---- bar charts: peak_ratio and var_ratio across arms ----------------------
order = list(piv.index)
x = np.arange(len(order))
n_m = len(SHAPE_LABELS)
w = 0.8 / n_m
palette_shape = ['steelblue', 'forestgreen', 'darkorange', 'mediumorchid', 'sienna']
fig, axes = plt.subplots(2, 1, figsize=(20, 11))
for ax, ratio, ref_lines in [
    (axes[0], 'peak_ratio', [1.0]),
    (axes[1], 'var_ratio',  [1.0, VAR_HI, VAR_LO]),
]:
    for i, label in enumerate(SHAPE_LABELS):
        offset = (i - (n_m - 1) / 2) * w
        ax.bar(x + offset, [by_method[label].loc[m, ratio] for m in order], w,
               label=label, color=palette_shape[i % len(palette_shape)], alpha=0.85)
    for rl in ref_lines:
        ax.axhline(rl, color='red', linestyle='--', alpha=0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{ratio}  (1.0 = preserved)')
    ax.set_ylabel(ratio)
    ax.legend(fontsize=8, ncol=n_m)
    ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Shape preservation per ablation arm (Stage 1 = pure-shift control)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nReading it: Stage 1 bars hug 1.0 (translation preserves shape). Every arm should also')
print('stay near 1.0 — if a reduced arm departs, the dropped loss WAS protecting 1D shape.')

## 8. kBET Evaluation

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

EVAL_LAYERS = [k for k, _ in ABL_LAYERS]
print(f'Evaluating {len(EVAL_LAYERS)} layers: {[ABL_LABEL[k] for k in EVAL_LAYERS]}')
print('\nReference points (same kBET pipeline):')
print('  Stage 1 (current, negative-peak):       0.6202')
print('  UniFORM:                                0.6315')
print('  GNN published config (w_adv=0.3):       0.709 +/- 0.006 (n=6)')
print('  DDPM K=1 (eval notebook):               0.7328 +/- 0.0033 (3 seeds)')
print('  its controls: S1+noise 0.6962, S1+shuffled 0.7025')

In [ ]:
import anndata as ad
import gc

all_kbet = {}
umap_coords = {}

# MEMORY NOTE: the original cell did `adata_kbet = adata_norm.copy()`, which duplicates
# EVERY layer (~141 MB each; 6 layers once the ablation arms exist = ~850 MB per
# iteration, 5 iterations). pegasus only ever reads .X here, so build a minimal
# AnnData instead: X + obs + var. That keeps the per-iteration cost at one 141 MB
# array rather than ~850 MB, and it is faster too.
for layer_name in EVAL_LAYERS:
    label = ABL_LABEL.get(layer_name, layer_name)
    print(f'\n{"-"*55}\n  {label}  ({layer_name})\n{"-"*55}')

    adata_kbet = ad.AnnData(
        X=np.asarray(adata_norm.layers[layer_name], dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}
        finally:
            del parts, adata_g, mmdata

    all_kbet[layer_name] = layer_res
    del adata_kbet
    gc.collect()

print('\nDone.')

In [ ]:
print('=' * 70)
print('kBET Summary (mean across groups — higher = better)')
print('=' * 70)
summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    if kbets:
        row = {'arm': ABL_LABEL.get(layer_name, layer_name), 'layer': layer_name,
               'mean_kBET': float(np.mean(kbets)), 'mean_chi2': float(np.mean(chi2s)),
               'n_groups': f'{len(kbets)}/{len(res)}'}
        summary_rows.append(row)
        print(f"  {row['arm']:>12s} ({layer_name:>15s})  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  ({row['n_groups']} groups)")
df_kbet = pd.DataFrame(summary_rows)

print('\n' + '=' * 70)
print('Per-group kBET')
print('=' * 70)
pergroup = pd.DataFrame({
    ABL_LABEL.get(layer_name, layer_name): {g: v['kBET'] for g, v in res.items()}
    for layer_name, res in all_kbet.items()
}).round(4)
print(pergroup.to_string())

if len(df_kbet):
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.bar(df_kbet['arm'], df_kbet['mean_kBET'], color='steelblue', alpha=0.85)
    ax.axhline(0.6315, color='red', linestyle='--', linewidth=1.3, label='UniFORM 0.6315')
    ax.axhline(0.709, color='black', linestyle=':', linewidth=1.3, label='GNN published 0.709')
    ax.set_ylabel('Mean kBET acceptance rate')
    ax.set_title('kBET — averaged DDPM vs GNN vs noise controls')
    ax.set_xticks(range(len(df_kbet)))
    ax.set_xticklabels(df_kbet['arm'], rotation=20, ha='right', fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
# Silhouette score — UniFORM-style: 3 cell types, computed PER SAMPLE
# Tumor epithelial (ECAD+), immune (CD45+), non-immune stromal (aSMA+)
#
# Why per-sample (not per clinical group): computing silhouette across two
# batch-paired samples lets the cross-batch offset inflate the raw baseline —
# good normalization then looks like it *lowers* silhouette (it just removed the
# batch-driven separation). Computing within each sample removes that confound:
# the score reflects only whether the 3 cell populations stay distinct.
# Labels are assigned once on raw data, fixed across all methods.

N_SIL = 3000           # cells per sample; O(n²) — stable well below this
MIN_TYPE_CELLS = 50    # a cell type counts as "present" if it has >= this many cells
                        # (decided on FULL per-sample counts → deterministic, seed-independent)

# Step 1: GMM thresholds for 3 anchor markers on full raw dataset (log1p space)
anchor_names = ['ECAD', 'CD45', 'aSMA']
anchor_thresholds = {}
print('Anchor marker GMM thresholds (log1p, raw data):')
for mname in anchor_names:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(
        X_log_preview[:, mi].reshape(-1, 1)
    )
    thr = float(np.mean(gmm.means_))
    anchor_thresholds[mname] = thr
    print(f'  {mname}: {thr:.3f}')

# Step 2: assign cell type labels on raw data — fixed for all methods
#   0 = tumor epithelial (ECAD+)
#   1 = immune           (CD45+)
#   2 = stromal          (aSMA+)
#  -1 = unassigned
# Priority for conflicts: ECAD > CD45 > aSMA
ecad_pos = X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']
cd45_pos = X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']
asma_pos = X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[asma_pos] = 2
cell_type[cd45_pos] = 1
cell_type[ecad_pos] = 0

print('\nCell type assignments (raw data):')
for code, label in [(0, 'tumor epithelial (ECAD+)'),
                    (1, 'immune (CD45+)'),
                    (2, 'stromal (aSMA+)'),
                    (-1, 'unassigned')]:
    n = (cell_type == code).sum()
    print(f'  {label}: {n:,}  ({100*n/len(cell_type):.1f}%)')

# Step 3: silhouette per sample, per method (no cross-batch mixing)
# Guard (deterministic): a sample is kept iff >= 2 cell types each have
# >= MIN_TYPE_CELLS cells on FULL data. Only those well-populated types are
# carried into the silhouette; sparse types (e.g. near-absent immune) are
# excluded from the computation rather than letting a subsample lottery drop
# the whole sample. This makes membership seed-independent and reproducible
# across notebooks.
print(f'\nSilhouette score per sample — 3 cell types in 20D log1p space (higher = better):')
sample_ids_all = adata_norm.obs['sample_id'].values
sil_rows = []
dropped = []
for sample in sorted(np.unique(sample_ids_all)):
    smask = sample_ids_all == sample
    s_labels = cell_type[smask]

    # full-data per-type counts → which types are substantial
    full_counts = {c: int((s_labels == c).sum()) for c in (0, 1, 2)}
    big = [c for c, n in full_counts.items() if n >= MIN_TYPE_CELLS]
    if len(big) < 2:
        dropped.append(sample)
        continue  # need >= 2 substantial cell types to define a silhouette

    # keep only the substantial types, then subsample for the O(n²) silhouette
    valid = np.where(np.isin(s_labels, big))[0]
    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sel_labels = s_labels[sel]

    sil_by_layer = {}
    for layer_name in ['raw'] + [k for k, _ in BIO_LAYERS]:
        if layer_name == 'raw':
            Xs = X_raw[smask]
        else:
            Xs = np.asarray(adata_norm.layers[layer_name])[smask]
        X_log = np.log1p(np.clip(Xs[sel], 0, None))
        sil_by_layer[layer_name] = round(
            float(silhouette_score(X_log, sel_labels, metric='euclidean')), 4
        )
    sil_rows.append({'sample': sample, **sil_by_layer})

sil_df = pd.DataFrame(sil_rows).set_index('sample')
sil_df.loc['Mean'] = sil_df.mean().round(4)
print(sil_df.to_string())
print(f'\n({len(sil_rows)} samples passed the guard: >=2 cell types with >={MIN_TYPE_CELLS} cells)')
if dropped:
    print(f'(dropped {len(dropped)}: {", ".join(dropped)})')
print('(higher = better — tumor / immune / stromal populations stay distinct after normalization)')


## 10. Verdict

**A — averaged DDPM (K = `K_MAIN`)**: the same seven tests as the eval notebook, except test A1
now compares against the GNN measured in this notebook (3 seeds) instead of the 0.7167 constant.
All pass → averaged DDPM is a headline candidate.

**B — GNN**: does the published GNN beat its own matched Gaussian control by more than the noise
threshold at every seed, beat the shuffled control, and stay deterministic per cell? If B1 fails,
the article's kBET gain is within noise of a matched blur and must be reported with that caveat.

In [ ]:
KBET_NOISE = 0.016   # ~1.4x the seed SD of an identical config
SIL_TOL    = 0.01
REF_POP    = 11      # Stage 1 / per-cell corrector markers within +/-5%

kb = {}
for layer_name, res in all_kbet.items():
    v = [r['kBET'] for r in res.values() if not np.isnan(r['kBET'])]
    kb[ABL_LABEL.get(layer_name, layer_name)] = float(np.mean(v)) if v else float('nan')


def mean_sd(labels):
    v = np.array([kb.get(l, np.nan) for l in labels])
    return float(np.nanmean(v)), float(np.nanstd(v, ddof=1))


def n_distorted(label):
    arm = shape_df[shape_df.method == label].set_index('marker').loc[uni_markers]
    return int(((arm.peak_ratio < 0.80) | (arm.var_ratio > 1.25) | (arm.var_ratio < 0.80)).sum())


L_DD = f'DDPM K{K_MAIN} r0'
L_GNN = f'GNN s{GNN_SEEDS[0]}'
dd_mean, dd_sd = mean_sd([f'DDPM K{K_MAIN} r{r}' for r in range(N_REPS)])
gnn_mean, gnn_sd = mean_sd([f'GNN s{s}' for s in GNN_SEEDS])
d_noise = kb.get(f'DDPM-K{K_MAIN} S1+noise', np.nan)
d_perm = kb.get(f'DDPM-K{K_MAIN} S1+shuffled', np.nan)
k1 = kb.get('DDPM K1', np.nan)
sil_s1 = float(sil_df.loc['Mean', 'normalized_base'])

print('kBET means:')
print(f'  DDPM K1 (single draw)       {k1:.4f}')
for K in K_LIST:
    if K != 1:
        print(f'  DDPM K{K} r0                 {kb.get(f"DDPM K{K} r0", np.nan):.4f}')
print(f'  DDPM K{K_MAIN} ({N_REPS} reps)           {dd_mean:.4f} +/- {dd_sd:.4f}   '
      f'(averaging cost vs K1: {dd_mean - k1:+.4f})')
print(f'  GNN (3 seeds)               {gnn_mean:.4f} +/- {gnn_sd:.4f}')

tests_A = [
    (f'A1 K{K_MAIN} kBET clearly above GNN', dd_mean - gnn_mean > KBET_NOISE,
     f'{dd_mean:.4f} vs GNN {gnn_mean:.4f}  (diff {dd_mean - gnn_mean:+.4f}, needs > +{KBET_NOISE})'),
    ('A2 beats both matched noise controls', dd_mean - max(d_noise, d_perm) > KBET_NOISE,
     f'S1+noise {d_noise:.4f}, S1+shuffled {d_perm:.4f}'),
    ('A3a 1D shape clean (0 unimodal distorted)', n_distorted(L_DD) == 0,
     f'{n_distorted(L_DD)} distorted'),
    ('A3b positive population >= Stage 1', int(summaries[L_DD]['pass_5pct'].sum()) >= REF_POP,
     f'{int(summaries[L_DD]["pass_5pct"].sum())}/20 within +/-5%'),
    ('A3c silhouette ~ Stage 1',
     float(sil_df.loc['Mean', f'ddpm_k{K_MAIN}_r0']) >= sil_s1 - SIL_TOL,
     f'{float(sil_df.loc["Mean", f"ddpm_k{K_MAIN}_r0"]):.4f} vs Stage 1 {sil_s1:.4f}'),
    ('A4a kBET stable across reps', dd_sd < KBET_NOISE, f'SD {dd_sd:.4f}'),
    ('A4b per-cell output mostly deterministic', STAB[K_MAIN]['ratio'] < 0.5,
     f'seed SD / |delta| = {STAB[K_MAIN]["ratio"]:.2f} (K=1: {STAB[1]["ratio"]:.2f})'),
]

paired = {s: kb.get(f'GNN s{s}', np.nan) - kb.get(f'GNN-s{s} S1+noise', np.nan) for s in GNN_SEEDS}
g_perm = kb.get(f'GNN-s{GNN_SEEDS[0]} S1+shuffled', np.nan)
tests_B = [
    ('B1 GNN beats matched S1+noise, every seed', all(v > KBET_NOISE for v in paired.values()),
     'paired diffs ' + ', '.join(f's{s} {v:+.4f}' for s, v in paired.items())),
    ('B2 GNN beats shuffled control (seed 0)', kb.get(L_GNN, np.nan) - g_perm > KBET_NOISE,
     f'{kb.get(L_GNN, np.nan):.4f} vs {g_perm:.4f}'),
    ('B3 GNN per-cell output deterministic', GNN_STAB < 0.5, f'seed SD / |delta| = {GNN_STAB:.4f}'),
    ('B4 GNN biology (shape / pos-pop / silhouette)',
     n_distorted(L_GNN) == 0 and int(summaries[L_GNN]['pass_5pct'].sum()) >= REF_POP
     and float(sil_df.loc['Mean', f'gnn_s{GNN_SEEDS[0]}']) >= sil_s1 - SIL_TOL,
     f'{n_distorted(L_GNN)} distorted, {int(summaries[L_GNN]["pass_5pct"].sum())}/20, '
     f'sil {float(sil_df.loc["Mean", f"gnn_s{GNN_SEEDS[0]}"]):.4f}'),
]

for title, tests in [(f'A. Averaged DDPM (K={K_MAIN})', tests_A), ('B. GNN (published config)', tests_B)]:
    print('\n' + '=' * 96)
    print(title)
    print('=' * 96)
    for name, ok, detail in tests:
        print(f'  {"PASS" if ok else "FAIL"}  {name:44s} {detail}')

okA = {n.split()[0]: ok for n, ok, _ in tests_A}
okB = {n.split()[0]: ok for n, ok, _ in tests_B}
print('\n' + '=' * 96)
if all(okA.values()):
    print(f'A: averaged DDPM (K={K_MAIN}) passes every test -> headline candidate: higher kBET than')
    print('   the GNN at the same biology, and deterministic enough per cell.')
elif okA['A4b'] and not okA['A1']:
    print(f'A: averaging fixed the per-cell randomness but the kBET lead over the GNN is gone ->')
    print('   much of single-draw DDPM\'s extra kBET was blur. Keep the GNN as the headline;')
    print('   DDPM is one arm of the comparison.')
else:
    print(f'A: failed {[n for n, ok in okA.items() if not ok]} -> DDPM stays a comparison arm.')
if all(okB.values()):
    print('B: the GNN clears its own matched noise controls -> its kBET gain is alignment, not blur.')
elif not okB['B1']:
    print('B: the GNN does NOT clearly beat a matched Gaussian blur at every seed -> the article must')
    print('   report this control and qualify the kBET claim.')
else:
    print(f'B: failed {[n for n, ok in okB.items() if not ok]}')